In [1]:
import itertools
import os
from copy import deepcopy
from importlib import resources as impresources


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u, constants as const
from scipy.optimize import curve_fit
from tqdm import trange
from tqdm.auto import tqdm


import mcfacts.vis.LISA as li
from mcfacts.fiducial_plots import linefunc
from mcfacts.fiducial_plots import make_gen_masks

from mcfacts.inputs.settings_manager import SettingsManager

from mcfacts.modules.accretion import ProgradeBlackHoleAccretion, BinaryBlackHoleAccretion
from mcfacts.modules.damping import ProgradeBlackHoleDamping, BinaryBlackHoleDamping
from mcfacts.modules.disk_capture import EvolveRetrogradeBlackHoles, RecaptureBinaryBlackHoles, \
    CaptureNSCProgradeBlackHoles
from mcfacts.modules.dynamics import SingleBlackHoleDynamics, BinaryBlackHoleDynamics, BinaryBlackHoleEccDynamics, \
    BinaryBlackHoleSpheroidDynamics, BinaryBlackHoleIonization
from mcfacts.modules.formation import BinaryBlackHoleFormation
from mcfacts.modules.gas_hardening import BinaryBlackHoleGasHardening
from mcfacts.modules.gw import InnerBlackHoleDynamics, BinaryBlackHoleEvolveGW
from mcfacts.modules.merge import ProcessBinaryBlackHoleMergers, ProcessEMRIMergers
from mcfacts.modules.migration import ProgradeBlackHoleMigration, BinaryBlackHoleMigration

from mcfacts.objects.actors import InitialBlackHoleReclassification, SingleBlackHoleRealityCheck, \
    BinaryBlackHoleRealityCheck, InnerDiskFilter, FlipRetroProFilter
from mcfacts.objects.agn_object_array import FilingCabinet
from mcfacts.objects.disk import AGNDisk
from mcfacts.objects.galaxy import Galaxy
from mcfacts.objects.populators import SingleBlackHolePopulator
from mcfacts.objects.snapshot import TxtSnapshotHandler, IniSnapshotHandler
from mcfacts.objects.timeline import SimulationTimeline, TimelineActor

from mcfacts.utilities.unit_conversion import si_from_r_g

from mcfacts.vis import data, styles
from mcfacts.vis import plotting


plt.style.use("mcfacts.vis.mcfacts_figures")

/Users/jakepost/PycharmProjects/McFACTS/src/mcfacts/modules/dynamics.py:548: SyntaxWarning: invalid escape sequence '\s'
  :math:`T_orb =2\\pi \sqrt(a^3/GM_{smbh})`
/Users/jakepost/PycharmProjects/McFACTS/src/mcfacts/modules/dynamics.py:1287: SyntaxWarning: invalid escape sequence '\s'
  :math:`T_orb =2\\pi \sqrt(a^3/GM_{smbh})`
/Users/jakepost/PycharmProjects/McFACTS/src/mcfacts/modules/dynamics.py:1935: SyntaxWarning: invalid escape sequence '\s'
  :math:`T_orb =2\\pi \sqrt(a^3/GM_{smbh})`
/Users/jakepost/PycharmProjects/McFACTS/src/mcfacts/modules/merge.py:454: SyntaxWarning: invalid escape sequence '\c'
  """Calculates the precessing spin component :math:`\chi_p` associated with a merger.


In [2]:
# Simulation Settings
RUNS_DIR = "./runs"
FIG_DIR = "./plots"
FIGSIZE = "apj_col"
W, H = plotting.set_size(FIGSIZE)
FORMATS = ("png", "pdf")
PAIR_COLORS = {"nostall": "tab:blue", "stall": "tab:red"}

GALAXY_NUM = 100
MIG_TORQUE_PRESCRIPTION = "jimenez_masset"
FRACTION_BIN_RETRO = 0.5

# Pull from GWTC-5
GAUSS_MU = 35
GAUSS_SIGMA = 2.3

ACTIVE_TIMESTEP_NUM = 500

NSC_BH_IMF_MODE = 2 #10

# m^-2 Initial mass function and gaussian at 35
INITIAL_MASS_FUNCTIONS = [
    #"default",
    "gaussian"
]

MU_CIRC_ENCOUNTER = [
    0.2,
    0.5,
    0.7,
    0.9
]

# Run 0.1 for stalling at 0.5
SIGMA_CIRC_ENCOUNTER = [
    0.025,
    0.1
]

# Finny and sigerton, ecc energy exchange at 20% in clusters.
MU_ECC_ENCOUNTER = [
    0.1,
    0.2
]

# Run 0.1 for stalling 0.5
SIGMA_ECC_ENCOUNTER = [
    0.02,
    0.1
]

# Build out the permutations of configs for each of the values we're studying.

def build_configs():
    configs = {}

    def add_config(overrides):
        static_settings = dict(
            galaxy_num=GALAXY_NUM,
            torque_prescription=MIG_TORQUE_PRESCRIPTION,
            fraction_bin_retro=FRACTION_BIN_RETRO,
            mass_pile_up=GAUSS_MU,
            nsc_imf_bh_mode=GAUSS_SIGMA,
            active_timestep_num=ACTIVE_TIMESTEP_NUM,
            flag_phenom_turb=False
        )
        combined_settings = static_settings | overrides

        gas = combined_settings["gas_hardening_prescription"]
        ts = combined_settings["active_timestep_num"]
        stall = combined_settings["stalling_separation"]
        mf = combined_settings["nsc_imf_bh_method"]
        mu_circ = combined_settings["mu_circ_encounter"]

        config_tag = f"{gas}_T-{ts}_stall-{stall}_imf-{mf}_mucirc-{mu_circ}"

        configs[config_tag] = SettingsManager(settings_overrides=combined_settings)

    for mu_circ_encounter in MU_CIRC_ENCOUNTER:
        add_config(
            dict(
                gas_hardening_prescription="analytical",
                stalling_separation=0,
                nsc_imf_bh_method="gaussian",
                mu_circ_encounter=mu_circ_encounter,
            )
        )

    return configs

CONFIGS = build_configs()

for i in CONFIGS.keys():
    print(i)

print(" ")
print(f"{len(CONFIGS)} configurations loaded, {(len(CONFIGS) * 18) / 60}m estimated run time")

analytical_T-500_stall-0_imf-gaussian_mucirc-0.2
analytical_T-500_stall-0_imf-gaussian_mucirc-0.5
analytical_T-500_stall-0_imf-gaussian_mucirc-0.7
analytical_T-500_stall-0_imf-gaussian_mucirc-0.9
 
4 configurations loaded, 1.2m estimated run time


In [3]:
# Custom timeline actor added to the simulation timeline to keep track of values not currently tracked by the canonical simulation.

class PopulationStateRecorder(TimelineActor):
    def __init__(self, history, galaxy_id, settings=None):
        super().__init__("Population State Recorder", settings)
        self.history = history
        self.galaxy_id = galaxy_id

    def __deepcopy__(self, memo):
        return self

    def perform(self, timestep, timestep_length, time_passed, filing_cabinet, agn_disk, random_generator):
        self.history.append({
            "galaxy_id": self.galaxy_id,
            "timestep": timestep,
            "filing_cabinet": deepcopy(filing_cabinet),
        })

In [4]:
# *** Main simulation script ***

CONFIG_SUBSET = CONFIGS #{}

#CONFIG_SUBSET["baruteau_stall-none_imf-default_circ-0p9_ecc-0p2"] = CONFIGS["baruteau_stall-none_imf-default_circ-0p9_ecc-0p2"]
#CONFIG_SUBSET["analytical_imf-default_circ-0p9_ecc-0p2"] = CONFIGS["analytical_imf-default_circ-0p9_ecc-0p2"]

AGN_DISK = AGNDisk(settings=list(CONFIG_SUBSET.values())[0])

RESULTS = {}

pbar = tqdm(CONFIG_SUBSET.items(), position=0)

for name, settings in pbar:
    pbar.set_description(name)

    out_dir = os.path.join(RUNS_DIR, name)

    population_cabinet = FilingCabinet()
    population_cabinet.ignore_consistency_check("blackholes_merged")
    population_cabinet.ignore_consistency_check("blackholes_lvk")

    history = []

    # Start galaxy loop
    for galaxy_id in trange(settings.galaxy_num):
        galaxy_seed = settings.seed - galaxy_id
        galaxy = Galaxy(seed=galaxy_seed, runs_folder=out_dir, galaxy_id=str(galaxy_id), settings=settings)

        galaxy.populate([SingleBlackHolePopulator()], AGN_DISK)

        pre_timeline = SimulationTimeline("Reclassification", timesteps=1, timestep_length=0)
        pre_timeline.add_timeline_actor(InitialBlackHoleReclassification())
        pre_timeline.add_timeline_actor(SingleBlackHoleRealityCheck())
        galaxy.run(pre_timeline, AGN_DISK)

        active_timeline = SimulationTimeline(
            "Active Timeline",
            timesteps=settings.active_timestep_num,
            timestep_length=galaxy.settings.active_timestep_duration_yr,
        )

        active_timeline.add_timeline_actor(SingleBlackHoleRealityCheck())

        population_recorder_actor = PopulationStateRecorder(history, galaxy_id, settings)
        active_timeline.add_timeline_actor(population_recorder_actor)

        prograde_array = galaxy.settings.bh_prograde_array_name
        innerdisk_array = galaxy.settings.bh_inner_disk_array_name
        inner_gw_only_array = galaxy.settings.bh_inner_gw_array_name

        active_timeline.add_timeline_actors([
            ProgradeBlackHoleMigration(target_array=innerdisk_array),
            ProgradeBlackHoleMigration(target_array=prograde_array),
            SingleBlackHoleRealityCheck(),

            ProgradeBlackHoleAccretion(target_array=innerdisk_array),
            ProgradeBlackHoleAccretion(target_array=prograde_array),
            ProgradeBlackHoleDamping(target_array=innerdisk_array),
            ProgradeBlackHoleDamping(target_array=prograde_array),

            EvolveRetrogradeBlackHoles(),
            SingleBlackHoleRealityCheck(),

            InnerBlackHoleDynamics(target_array=innerdisk_array),
            InnerBlackHoleDynamics(target_array=inner_gw_only_array),
            SingleBlackHoleDynamics(target_array=innerdisk_array),
            SingleBlackHoleDynamics(target_array=prograde_array),
        ])

        active_timeline.add_timeline_actors([
            BinaryBlackHoleDamping(),

            BinaryBlackHoleDynamics(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleEccDynamics(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleGasHardening(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleAccretion(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleSpheroidDynamics(reality_merge_checks=False),
            ProcessBinaryBlackHoleMergers(),

            RecaptureBinaryBlackHoles(),
            BinaryBlackHoleMigration(),
            BinaryBlackHoleRealityCheck(),

            BinaryBlackHoleEvolveGW(),

            BinaryBlackHoleIonization(),
            ProcessBinaryBlackHoleMergers(),

            BinaryBlackHoleFormation()
        ])

        active_timeline.add_timeline_actor(CaptureNSCProgradeBlackHoles())
        active_timeline.add_timeline_actor(InnerDiskFilter())
        active_timeline.add_timeline_actor(FlipRetroProFilter())
        active_timeline.add_timeline_actor(ProcessEMRIMergers())

        galaxy.run(active_timeline, AGN_DISK)

        history.append({
            "galaxy_id": galaxy_id,
            "timestep": settings.active_timestep_num,
            "filing_cabinet": galaxy.filing_cabinet,
        })

        population_arrays = {
            "blackholes_merged": [settings.bbh_merged_array_name],
            "blackholes_lvk": [settings.bbh_gw_array_name],
            "blackholes_ejected": [settings.bh_ejected_array_name],
            "blackholes_emri": [settings.bh_inner_disk_array_name, settings.bh_inner_gw_array_name, settings.emri_array_name],
        }

        for key, value in population_arrays.items():
            for object_array_name in value:
                if object_array_name not in galaxy.filing_cabinet:
                    continue

                object_array = galaxy.filing_cabinet.get_array(object_array_name)
                object_array.galaxy_id = np.full(len(object_array.unique_id), galaxy_id)

                population_cabinet.create_or_append_array(key, object_array)
    # End galaxy loop

    # Create the output directory for this run
    os.makedirs(out_dir, exist_ok=True)

    # Save run files
    TxtSnapshotHandler(settings=settings).save_cabinet(out_dir, "population", population_cabinet)
    IniSnapshotHandler(settings=settings).save_settings(out_dir, "settings", settings)
    pd.DataFrame(history).to_csv(os.path.join(out_dir, "history.csv"), index=False)

    RESULTS[name] = {
        "out_dir": out_dir,
        "settings": settings,
        "cabinet": population_cabinet,
        "history": pd.DataFrame(history),
    }

  0%|          | 0/4 [00:00<?, ?it/s]


100%|██████████| 100/100 [04:13<00:00,  2.54s/it]

100%|██████████| 100/100 [03:59<00:00,  2.39s/it]

100%|██████████| 100/100 [04:05<00:00,  2.45s/it]

100%|██████████| 100/100 [03:54<00:00,  2.35s/it]


In [5]:
# Show structure of the RESULTS object
def print_results_structure(results, indent=0):
    for key, value in results.items():
        if isinstance(value, dict):
            print(" " * indent + f"{key}:")
            print_results_structure(value, indent + 4)
        else:
            print(" " * indent + f"{key}: {type(value).__name__}")

print_results_structure(RESULTS)

analytical_T-500_stall-0_imf-gaussian_mucirc-0.2:
    out_dir: str
    settings: SettingsManager
    cabinet: FilingCabinet
    history: DataFrame
analytical_T-500_stall-0_imf-gaussian_mucirc-0.5:
    out_dir: str
    settings: SettingsManager
    cabinet: FilingCabinet
    history: DataFrame
analytical_T-500_stall-0_imf-gaussian_mucirc-0.7:
    out_dir: str
    settings: SettingsManager
    cabinet: FilingCabinet
    history: DataFrame
analytical_T-500_stall-0_imf-gaussian_mucirc-0.9:
    out_dir: str
    settings: SettingsManager
    cabinet: FilingCabinet
    history: DataFrame


In [17]:
for name, res in RESULTS.items():

    arr = res["history"].loc[:, ['filing_cabinet']].to_numpy()
    ge = res["history"].loc[:, ['galaxy_id']].to_numpy().squeeze()
    te = res["history"].loc[:, ['timestep']].to_numpy().squeeze()
    new_arr = arr.squeeze()

    machs = np.array([])
    masses = np.array([])
    uuids = np.array([])
    gens = np.array([])

    for r, g, t in zip(new_arr, ge, te):
        if "blackholes_binary" not in r:
            continue

        bins = r.get_array("blackholes_binary")

        is_new = ~np.isin(bins.unique_id, uuids)

        bin_sep = bins.bin_sep[is_new]
        si_unit_bin_sep = si_from_r_g(AGN_DISK.settings.smbh_mass, bin_sep)

        mass_1 = (bins.mass[is_new] * u.solMass).to(u.kg)
        mass_2 = (bins.mass_2[is_new] * u.solMass).to(u.kg)
        mass_tot = mass_1 + mass_2

        orb_vel_1 = np.sqrt((const.G * (mass_1 + mass_2) / si_unit_bin_sep))

        sp = AGN_DISK.disk_sound_speed(bins.orb_a[is_new]) * u.m/u.s

        mach = orb_vel_1 / sp

        machs = np.append(machs, mach.value)
        masses = np.append(masses, (np.minimum(mass_1.value, mass_2.value) / np.maximum(mass_1.value, mass_2.value)))
        uuids = np.append(uuids, bins.unique_id[is_new])
        gens = np.append(gens, bins.gen[is_new])


    merged = res["cabinet"].get_array("blackholes_merged")
    cm = np.isin(uuids, merged.unique_id)


    fig = plt.figure(dpi=600)

    plt.vlines([1], ymin=0, ymax=1, ls="--", color="black", label="$\\mathcal{M} = 1$", linewidth=1)

    plt.scatter(machs[cm][gens[cm] == 1], masses[cm][gens[cm] == 1], c="darkgoldenrod", s=8, marker="*", label="g1 merged")
    plt.scatter(machs[cm][gens[cm] == 2], masses[cm][gens[cm] == 2], c="rebeccapurple", s=8, marker="*", label="g2 merged")
    plt.scatter(machs[cm][gens[cm] > 2], masses[cm][gens[cm] > 2], c="red", s=8, marker="*", label="g3+ merged")


    plt.scatter(machs[~cm][gens[~cm] == 1], masses[~cm][gens[~cm] == 1], edgecolors="darkgoldenrod", s=8, marker="o", facecolors='none', label="g1 held")
    plt.scatter(machs[~cm][gens[~cm] == 2], masses[~cm][gens[~cm] == 2], edgecolors="rebeccapurple", s=8, marker="o", facecolors='none', label="g2 held")
    plt.scatter(machs[~cm][gens[~cm] > 2], masses[~cm][gens[~cm] > 2], edgecolors="red", s=8, marker="o", facecolors='none', label="g3+ held")

    #plt.xticks([0, 1, 2, 3, 4, 5, 6, 7, 8])

    plt.title(name)
    plt.ylabel("q ($m_2/m_1$)")
    plt.xlabel("$\\mathcal{M}$")
    plt.xscale("log")
    plt.ylim((0, 1))
    plt.legend()

    plt.show()
    plt.close()

KeyboardInterrupt: 

In [14]:
GENS = ("gen1", "gen2", "genX")
GEN_LABELS = ('1g-1g', '2g-1g or 2g-2g', r'$\geq$3g-Ng')
GEN_COLORS = [getattr(styles, f"color_{g}") for g in GENS]
GEN_STYLE = {1: ("g1", "darkgoldenrod"), 2: ("g2", "rebeccapurple"), 3: ("g3+", "red")}

LISA_FREQ = np.logspace(-5, 0, 1000)
LISA_HC = np.sqrt(LISA_FREQ * li.LISA().Sn(LISA_FREQ))
LIGO = {det: pd.read_csv(impresources.files(data) / fname, sep='\t', header=None) for det, fname in {
    "H1": 'O3-H1-C01_CLEAN_SUB60HZ-1262197260.0_sensitivity_strain_asd.txt',
    "L1": 'O3-L1-C01_CLEAN_SUB60HZ-1240573680.0_sensitivity_strain_asd.txt'}.items()}


# Component Drawing Methods
def draw_scatter_by_gen(ax, masks, x, y, scale=1.0):
    for mask, label, g in zip(masks, GEN_LABELS, GENS):
        ax.scatter(x[mask], y[mask], s=scale * getattr(styles, f"markersize_{g}"),
                   marker=getattr(styles, f"marker_{g}"), edgecolor=getattr(styles, f"color_{g}"),
                   facecolor="none", alpha=getattr(styles, f"markeralpha_{g}"), label=label)


def draw_hist_by_gen(ax, masks, x, bins, **kw):
    ax.hist([x[mask] for mask in masks], bins=bins, color=GEN_COLORS, label=GEN_LABELS,
            align='left', alpha=0.9, rwidth=0.8, stacked=True, **kw)


def draw_trap_line(ax, s):
    ax.axvline(s.disk_radius_trap, color='k', ls='--', zorder=0,
               label=f'Trap Radius = {s.disk_radius_trap:.0f} $R_g$')


# Plot Methods
def plot_mach_q(ax, result, gen_mask):
    machs, qs, gens, uuids = [], [], [], np.array([])

    for res in result["history"]["filing_cabinet"]:
        if "blackholes_binary" not in res:
            continue

        bins = res.get_array("blackholes_binary")
        new = ~np.isin(bins.unique_id, uuids)

        sep = si_from_r_g(AGN_DISK.settings.smbh_mass, bins.bin_sep[new])
        m_tot = ((bins.mass[new] + bins.mass_2[new]) * u.solMass).to(u.kg)
        c_s = AGN_DISK.disk_sound_speed(bins.orb_a[new]) * u.m / u.s

        machs.append((np.sqrt(const.G * m_tot / sep) / c_s).decompose().value)
        qs.append(np.minimum(bins.mass[new], bins.mass_2[new]) / np.maximum(bins.mass[new], bins.mass_2[new]))
        gens.append(bins.gen[new])
        uuids = np.append(uuids, bins.unique_id[new])

    machs, qs, gens = (np.concatenate(x or [[]]) for x in (machs, qs, gens))
    merged = np.isin(uuids, result["cabinet"].get_array("blackholes_merged").unique_id)

    gens = np.clip(gens, 1, 3)

    ax.axvline(1, ls="--", color="black", lw=1, label=r"$\mathcal{M} = 1$")

    for g, (tag, color) in GEN_STYLE.items():
        m, h = (gens == g) & merged, (gens == g) & ~merged
        ax.scatter(machs[m], qs[m], c=color, s=8, marker="*", label=f"{tag} merged")
        ax.scatter(machs[h], qs[h], edgecolors=color, facecolors="none", s=8, marker="o", label=f"{tag} held")

    ax.set_xscale("log")
    ax.set_ylim(0, 1)


def plot_remnant_mass_hist(ax, result, gen_mask):
    bh_merged = result["cabinet"].get_array("blackholes_merged")

    draw_hist_by_gen(ax, gen_mask, bh_merged.mass_final, np.arange(int(bh_merged.mass_final.min()), int(bh_merged.mass_final.max()) + 2))
    n = sum(mask.sum() for mask in gen_mask)

    for k, (mask, c) in enumerate(zip(gen_mask, GEN_COLORS)):
        ax.text(0.97, 0.95 - 0.08 * k, f"{100 * mask.sum() / n:.1f}%", color=c,
                ha="right", va="top", fontsize=5, transform=ax.transAxes)

    ax.set(xlabel=r'Remnant Mass [$M_\odot$]', ylabel='Number of Mergers')


def plot_mass_v_radius(ax, result, gen_mask):
    bh_merged, sm = result["cabinet"].get_array("blackholes_merged"), result["settings"]

    draw_scatter_by_gen(ax, gen_mask, bh_merged.orb_a, bh_merged.mass_final)
    draw_trap_line(ax, sm)

    ax.set(xlabel=r'Radius [$R_g$]', ylabel=r'Remnant Mass [$M_\odot$]', xscale='log', yscale='log', ylim=(15, 1000))


def plot_q_chi_eff(ax, result, gen_mask):
    bh_merged = result["cabinet"].get_array("blackholes_merged")

    q = np.minimum(bh_merged.mass, bh_merged.mass_2) / np.maximum(bh_merged.mass, bh_merged.mass_2)

    draw_scatter_by_gen(ax, gen_mask, bh_merged.chi_eff, q)

    x = np.array([-1.0, 1.0])
    fits = ((gen_mask[1] | gen_mask[2], 'dashed', 'gray', r'$\geq$2g'), (np.ones_like(q, bool), 'solid', 'black', 'all'))

    for k, (sel, ls, c, tag) in enumerate(fits):
        if sel.sum() > 1:
            (slope,), cov = curve_fit(linefunc, q[sel], bh_merged.chi_eff[sel])
            ax.plot(linefunc(x, slope), x, ls=ls, lw=1, color=c, zorder=3, label=f'fit ({tag})')
            ax.text(0.03, 0.03 + 0.08 * k, rf'$d\chi/dq$ ({tag}) = {slope:.2f}$\pm${np.sqrt(cov[0, 0]):.2f}',
                    color=c, fontsize=5, transform=ax.transAxes)

    ax.set(xlabel=r'$\chi_{\rm eff}$', ylabel=r'$q = M_2/M_1$', xlim=(-1, 1), ylim=(0, 1))


def plot_chi_p_v_radius(ax, result, gen_mask):
    bh_merged, sm = result["cabinet"].get_array("blackholes_merged"), result["settings"]

    draw_scatter_by_gen(ax, gen_mask, bh_merged.orb_a, bh_merged.chi_p)
    draw_trap_line(ax, sm)

    ax.set(xlabel=r'Radius [$R_g$]', ylabel=r'$\chi_{\rm p}$', xscale='log', ylim=(0, 1))


def plot_time_of_merger(ax, result, gen_mask):
    bh_merged = result["cabinet"].get_array("blackholes_merged")

    draw_scatter_by_gen(ax, gen_mask, bh_merged.time_merged / 1e6, bh_merged.mass_final)

    ax.set(xlabel='Time [Myr]', ylabel=r'Remnant Mass [$M_\odot$]', yscale='log', ylim=(17, 180))


def plot_m1m2(ax, result, gen_mask):
    bh_merged = result["cabinet"].get_array("blackholes_merged")

    draw_scatter_by_gen(ax, gen_mask, np.maximum(bh_merged.mass, bh_merged.mass_2), np.minimum(bh_merged.mass, bh_merged.mass_2))

    ax.set(xlabel=r'$M_1$ [$M_\odot$]', ylabel=r'$M_2$ [$M_\odot$]', xscale='log', yscale='log',
           xlim=(9, 110), ylim=(9, 110))


def plot_spin_v_mass(ax, results, gen_mask):
    bh_merged = results["cabinet"].get_array("blackholes_merged")

    draw_scatter_by_gen(ax, gen_mask, bh_merged.mass_final, bh_merged.spin_final)

    ax.set(xlabel=r'Remnant Mass [$M_\odot$]', ylabel=r'$a_{\rm final}$', xscale='log', xlim=(9, 140), ylim=(0.41, 1.01))


def plot_spin_v_radius(ax, result, gen_mask):
    bh_merged, sm = result["cabinet"].get_array("blackholes_merged"), result["settings"]

    draw_scatter_by_gen(ax, gen_mask, bh_merged.orb_a, bh_merged.spin_final)
    draw_trap_line(ax, sm)

    ax.set(xlabel=r'Radius [$R_g$]', ylabel=r'$a_{\rm final}$', xscale='log')


def plot_strain_lisa(ax, result, gen_mask):
    cabinet, sm = result["cabinet"], result["settings"]
    lvk, emri = cabinet.get_array("blackholes_lvk"), cabinet.get_array("blackholes_emri")

    ax.loglog(LISA_FREQ, LISA_HC, color='tab:orange', lw=1, zorder=0, label='LISA Sensitivity')

    ax.scatter(emri.gw_freq, emri.gw_char_strain, s=0.4 * styles.markersize_gen1, alpha=styles.markeralpha_gen1, label='EMRI')

    draw_scatter_by_gen(ax, make_gen_masks(lvk.gen, lvk.gen_2), lvk.gw_freq, lvk.gw_char_strain, scale=0.4)

    if sm.stalling_separation > 0:
        a = sm.stalling_separation * const.G * sm.smbh_mass * const.M_sun / const.c ** 2
        f = (np.sqrt(const.G * (lvk.mass + lvk.mass_2) * const.M_sun / a ** 3) / np.pi).value

        ax.axvspan(f.min(), f.max(), color='red', alpha=0.3, label='Stalling Range')
        ax.axvline(f.mean(), color='red', ls='--', lw=1, label='Stalling Avg.')

    ax.set(xlabel=r'$\nu_{\rm GW}$ [Hz]', ylabel=r'$h_{\rm char}$', xlim=(5e-8, 10), ylim=(1e-26, 1e-15))


def plot_strain_lvk(ax, result, gen_mask):
    lvk = result["cabinet"].get_array("blackholes_lvk")

    for (det, df), c in zip(LIGO.items(), ('tab:blue', 'tab:orange')):
        ax.loglog(df[0], df[1], color=c, lw=1, zorder=0, label=f'LIGO O3, {det} Sens.')

    draw_scatter_by_gen(ax, make_gen_masks(lvk.gen, lvk.gen_2), lvk.gw_freq, lvk.gw_strain, scale=0.4)

    ax.set(xlabel=r'$\nu_{\rm GW}$ [Hz]', ylabel=r'$h_0$', xlim=(5, 1e4), ylim=(2e-24, 1e-19))


def plot_grid(results, draw_function, row_data, col_data, row_format=str, col_format=str, **criteria):
    (row_key, row_vals), (col_key, col_vals) = row_data, col_data

    fig, ax = plt.subplots(len(row_vals), len(col_vals), sharex=True, sharey=True, squeeze=False,
                             figsize=(W * len(col_vals), H), layout="constrained")

    for (i, rv), (j, cv) in itertools.product(enumerate(row_vals), enumerate(col_vals)):
        runs = select_with_settings(results, **criteria, **{row_key: rv, col_key: cv})

        if len(runs) != 1:
            ax[i, j].text(0.5, 0.5, f"{len(runs)} matching runs", ha="center", va="center", transform=ax[i, j].transAxes)
            continue

        selected_results = results[runs[0]]
        bh_merged = selected_results["cabinet"].get_array("blackholes_merged")
        gen_mask = make_gen_masks(bh_merged.gen, bh_merged.gen_2)

        draw_function(ax[i, j], selected_results, gen_mask)

        ax[i, j].grid(True, color='gray', ls='dashed')
        ax[i, j].set_axisbelow(True)

    for axis in ax.flat:
        axis.label_outer()

    for axis, cv in zip(ax[0], col_vals):
        axis.set_title(col_format(cv))

    for axis, rv in zip(ax[:, 0], row_vals):
        axis.set_ylabel(f"{row_format(rv)}\n{axis.get_ylabel()}")

    handles = {label: handle for axis in ax.flat for handle, label in zip(*axis.get_legend_handles_labels())}

    fig.legend(handles.values(), handles.keys(), loc="outside upper center", ncols=4)

    return fig, ax

         	Approximation will be used...


In [ ]:
GRID = dict(
    row_data=("gas_hardening_prescription", [*GAS_HARDENING_PRESCRIPTIONS, "analytical"]),
    col_data=("stalling_separation", STALLING_VALUES),
    nsc_imf_bh_method="default",
    mu_circ_encounter=0.9,
    mu_ecc_encounter=0.1,
    col_format=lambda v: rf"{STALLING_LABELS[v]}"
)

PLOTS = {
    "mach_q":               plot_mach_q,
    "merger_remnant_mass":  plot_remnant_mass_hist,
    "merger_mass_v_radius": plot_mass_v_radius,
    "q_chi_eff":            plot_q_chi_eff,
    "r_chi_p":              plot_chi_p_v_radius,
    "time_of_merger":       plot_time_of_merger,
    "m1m2":                 plot_m1m2,
    "spin_v_mass":          plot_spin_v_mass,
    "spin_vs_radius":       plot_spin_v_radius,
    "gw_strain_lisa":       plot_strain_lisa,
    "gw_strain_lvk":        plot_strain_lvk,
}

os.makedirs(FIG_DIR, exist_ok=True)

for name, draw_func in PLOTS.items():
    fig, _ = plot_grid(RESULTS, draw_function=draw_func, **GRID)

    for ext in FORMATS:
        fig.savefig(f"{FIG_DIR}/{name}_grid.{ext}",)
    plt.show(fig)
    plt.close(fig)

In [ ]:
GRID = dict(
    row_data=("gas_hardening_prescription", [*GAS_HARDENING_PRESCRIPTIONS, "analytical"]),
    col_data=("active_timestep_num", [100, 200, 300, 400, 500]),
    nsc_imf_bh_method="gaussian",
    mu_circ_encounter=0.9,
    mu_ecc_encounter=0.2,
    col_format=lambda v: rf"{v / 100} $Myr$"
)

PLOTS = {
    #"mach_q":               plot_mach_q,
    "merger_remnant_mass":  plot_remnant_mass_hist,
    "merger_mass_v_radius": plot_mass_v_radius,
    "q_chi_eff":            plot_q_chi_eff,
    "r_chi_p":              plot_chi_p_v_radius,
    "time_of_merger":       plot_time_of_merger,
    "m1m2":                 plot_m1m2,
    "spin_v_mass":          plot_spin_v_mass,
    "spin_vs_radius":       plot_spin_v_radius,
    "gw_strain_lisa":       plot_strain_lisa,
    "gw_strain_lvk":        plot_strain_lvk,
}

os.makedirs(FIG_DIR, exist_ok=True)

for name, draw_func in PLOTS.items():
    fig, _ = plot_grid(RESULTS, draw_function=draw_func, **GRID)

    for ext in FORMATS:
        fig.savefig(f"{FIG_DIR}/{name}_grid.{ext}",)
    plt.show(fig)
    plt.close(fig)

In [ ]:
def object_array_to_df(result, key):
    """One population array -> DataFrame, with galaxy provenance attached."""
    population_cabinet = result["cabinet"]

    if key not in population_cabinet.agn_objects:
        return pd.DataFrame()

    object_array = population_cabinet.agn_objects[key]
    df = pd.DataFrame({k: np.asarray(v) for k, v in object_array.get_super_dict().items()})

    if not df.empty and {"mass", "mass_2"}.issubset(df.columns):
        m1 = df["mass"].to_numpy(dtype=float)
        m2 = df["mass_2"].to_numpy(dtype=float)
        df["mass_ratio"] = np.minimum(m1, m2) / np.maximum(m1, m2)

    return df

In [ ]:
summary_rows = []
for name, result in RESULTS.items():
    merged = object_array_to_df(result, "blackholes_merged")

    galaxy_num = result["settings"].galaxy_num

    per_galaxxy = merged.galaxy_id.value_counts()

    summary_rows.append({
        "name": name,
        "n_galaxy": galaxy_num,
        "n_merged": len(merged),
        "n_lvk": len(object_array_to_df(result, "blackholes_lvk")),
        "n_emri": len(object_array_to_df(result, "blackholes_emri")),
        "n_ejected": len(object_array_to_df(result, "blackholes_ejected")),
        "mergers_per_galaxy": per_galaxy.mean(),
        "mergers_per_galaxy_std": per_galaxy.std(),
    })

SUMMARY = pd.DataFrame(summary_rows)

SUMMARY.set_index('name', inplace=True)

os.makedirs(FIG_DIR, exist_ok=True)
SUMMARY.to_csv(os.path.join(FIG_DIR, "summary_table.csv"), index=False)

SUMMARY

In [15]:
def save_figure(fig, name, fig_dir=FIG_DIR):
    os.makedirs(fig_dir, exist_ok=True)
    paths = []

    for ext in FORMATS:
        path = os.path.join(fig_dir, f"{name}.{ext}")
        fig.savefig(path, format=ext, bbox_inches="tight",
                    dpi=300 if ext == "png" else None)
        paths.append(path)

    return paths


def select_with_settings(results, **criteria):
    out = []

    for run_name, result in results.items():
        result_settings = result["settings"].settings_finals

        if all(result_settings.get(key) == value for key, value in criteria.items()):
            out.append(run_name)

    return out

In [ ]:
select_with_settings(
            RESULTS,
            gas_hardening_prescription="analytical",
            nsc_imf_bh_method="default",
            mu_circ_encounter=0.9,
            mu_ecc_encounter=0.1
        )

In [ ]:
from matplotlib.legend_handler import HandlerErrorbar
from matplotlib.container import ErrorbarContainer

# Mergers vs Stalling
fig, (ax, ax_var) = plt.subplots(1, 2, figsize=(W * 1.45, H * 1.2), sharey=True, gridspec_kw={"width_ratios": [4, 1], "wspace": 0.05})

for presc, color in zip(["baruteau", "stahler"], ["tab:blue", "tab:red"]):
    for imf in INITIAL_MASS_FUNCTIONS[:1]:
        rows = {
            f"{STALLING_TAGS[s]}": select_with_settings(
                RESULTS,
                stalling_separation=s,
                gas_hardening_prescription=presc,
                nsc_imf_bh_method=imf,
                mu_circ_encounter=0.9,
                mu_ecc_encounter=0.1

            )
            for s in STALLING_VALUES
        }

        y = []
        e = []

        for stalling_value in STALLING_VALUES[:4]:
            for run_name in rows[STALLING_TAGS[stalling_value]]:
                sum_data = SUMMARY.loc[run_name]
                y.append(sum_data.mergers_per_galaxy)
                e.append(sum_data.mergers_per_galaxy_std / np.sqrt(sum_data.n_galaxy))
                break

        ax.errorbar(STALLING_VALUES[:4], y, yerr=e, ls="-", marker="o", ms=4, capsize=2, lw=1, color=color)

        v = select_with_settings(
            RESULTS,
            stalling_separation=-1,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method=imf,
            mu_circ_encounter=0.9,
            mu_ecc_encounter=0.1
        )

        v_data = SUMMARY.loc[v]
        vy = v_data.mergers_per_galaxy
        ve = v_data.mergers_per_galaxy_std / np.sqrt(sum_data.n_galaxy)

        ax_var.errorbar([0], [vy], yerr=[ve], ls="none", marker="o", ms=4, capsize=2, color=color)

w = select_with_settings(
    RESULTS,
    stalling_separation=0,
    gas_hardening_prescription="analytical",
    nsc_imf_bh_method="default",
    mu_circ_encounter=0.9,
    mu_ecc_encounter=0.1
)

w_data = SUMMARY.loc[w]
wy = w_data.mergers_per_galaxy
we = w_data.mergers_per_galaxy_std / np.sqrt(sum_data.n_galaxy)

ax_var.errorbar([1], [wy], yerr=[we], ls="none", marker="o", ms=4, capsize=2, color="tab:purple")

ax.errorbar([], [], yerr=[], ls="none", marker="o", ms=4, capsize=2, color="tab:blue", label="baruteau, default IMF")
ax.errorbar([], [], yerr=[], ls="none", marker="o", ms=4, capsize=2, color="tab:red", label="stahler, default IMF")
ax.errorbar([], [], yerr=[], ls="none", marker="o", ms=4, capsize=2, color="tab:purple", label="analytical, default IMF")


ax.set_xlabel(r"stalling separation [$r_g$]")
ax.set_ylabel("mergers per galaxy")

#ax.axvline(2.0, color="grey", ls=":", lw=1)
#ax.text(2.05, ax.get_ylim()[1] * 0.96, "GW tracking\nthreshold", fontsize=5, color="grey", va="top")

ax.legend(fontsize=5, frameon=False, handler_map={ErrorbarContainer: HandlerErrorbar(xerr_size=0, yerr_size=0)}, numpoints=1)

ax_var.set_xticks([0, 1], [r"variable" "\n" r"($\mathcal{M}=1$)", r"variable" "\n" r"(analytical)"], fontsize=5)
ax_var.tick_params(axis="y", left=False)
ax_var.set_xlim(-0.5, 1.5)

fig.suptitle("Number of Mergers vs Stalling Separation", fontsize=7)

save_figure(fig, "mergers_vs_stalling", FIG_DIR)

In [ ]:
def stall_color(stall, stalling_values):
    if stall == -1.0:
        return "black"

    exclude_var = dict([(s, i) for i, s in enumerate(stalling_values) if s >= 0])
    index = exclude_var[stall]

    return plt.cm.tab10(index)

fig = plt.figure(figsize=(W, H * 1.2))
ax = fig.add_subplot()

for presc, ls in zip(("baruteau", "stahler"), ("-", "--")):
    rows = {
        f"{STALLING_TAGS[s]}": select_with_settings(
            RESULTS,
            stalling_separation=s,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method="default",
            mu_circ_encounter=0.9,
            mu_ecc_encounter=0.1

        )
        for s in STALLING_VALUES
    }

    for s_value in STALLING_VALUES:
        row_name = rows[STALLING_TAGS[s_value]][0]
        if row_name == "":
            continue

        res = RESULTS[row_name]

        df = object_array_to_df(res, "blackholes_merged")

        if not len(df):
            continue

        t = df["time_merged"].value_counts().sort_index().cumsum()
        tk = t.index
        tv = t.values

        c = stall_color(s_value, STALLING_VALUES)
        l = ("variable" if s_value == -1 else f"" if s_value == 0 else f"{s_value:g}")

        ax.plot(tk / 1000000, tv, lw=0.7, color=c, label=f"{presc} {l}", ls=ls)

w = select_with_settings(
    RESULTS,
    stalling_separation=0,
    gas_hardening_prescription="analytical",
    nsc_imf_bh_method="default",
    mu_circ_encounter=0.9,
    mu_ecc_encounter=0.1
)

res = RESULTS[w[0]]

df = object_array_to_df(res, "blackholes_merged")

t = df["time_merged"].value_counts().sort_index().cumsum()
tk = t.index
tv = t.values

ppl, = ax.plot(tk / 1000000, tv, lw=1, color="tab:purple", label=f"_nolegend_", ls=":")

ax.set_xticks([0, .10, .20, .30, .40, .50, .60, .70])
ax.set_xlabel("time [Myr]", fontsize=7)
ax.set_ylabel("cumulative mergers\nper galaxy", fontsize=7)

second_legend = ax.legend(handles=[ppl], labels=["analytical"], fontsize=4.5, frameon=False, title_fontsize=4.5, bbox_to_anchor=(0.4, 0.7))

ax.legend(fontsize=4.5, frameon=False, title=r"stall [$r_g$]", title_fontsize=4.5, ncol=2)

ax.add_artist(second_legend)

fig.suptitle("Cumulative Mergers vs Time", fontsize=7)

In [ ]:
def stall_color(stall, stalling_values):
    if stall == -1.0:
        return "black"

    exclude_var = dict([(s, i) for i, s in enumerate(stalling_values) if s >= 0])
    index = exclude_var[stall]

    return plt.cm.tab10(index)

fig = plt.figure(figsize=(W, H * 1.2))
ax = fig.add_subplot()

for presc, ls in zip(("baruteau", "stahler", "analytical"), ("-", "--", ":")):
    rows = {
        f"{CIRC_HARDENING_TAGS[s]}": select_with_settings(
            RESULTS,
            stalling_separation=0 if presc == "analytical" else 1,
            mu_circ_encounter=s,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method="default",
            mu_ecc_encounter=0.1
        )
        for s in CIRC_HARDENING_MU
    }

    for s_value in CIRC_HARDENING_MU:
        row_name = rows[CIRC_HARDENING_TAGS[s_value]][0]
        if row_name == "":
            continue

        res = RESULTS[row_name]

        df = object_array_to_df(res, "blackholes_merged")

        if not len(df):
            continue

        t = df["time_merged"].value_counts().sort_index().cumsum()
        tk = t.index
        tv = t.values

        c = stall_color(s_value, CIRC_HARDENING_MU)
        l = ("variable" if s_value == -1 else f"" if s_value == 0 else f"{s_value:g}")

        ax.plot(tk / 1000000, tv, lw=0.7, color=c, label=f"{presc} {l}", ls=ls)

ax.set_xticks([0, .10, .20, .30, .40, .50, .60, .70])
ax.set_xlabel("time [Myr]", fontsize=7)
ax.set_ylabel("cumulative mergers\nper galaxy", fontsize=7)

ax.legend(fontsize=4.5, frameon=False, title=r"Circ Encounter $\Delta E$", title_fontsize=4.5, ncol=3)
#ax.set_yscale("log")

fig.suptitle("Cumulative Mergers vs Time", fontsize=7)

#stall sep =1

In [ ]:
# Max Remnant Mass vs Time

fig = plt.figure(figsize=(W, H * 1.2))
ax = fig.add_subplot()

for presc, ls in zip(("baruteau", "stahler"), ("-", "--")):
    rows = {
        f"{STALLING_TAGS[s]}": select_with_settings(
            RESULTS,
            stalling_separation=s,
            gas_hardening_prescription=presc,
            nsc_imf_bh_method="default",
            mu_circ_encounter=0.9,
            mu_ecc_encounter=0.1

        )
        for s in STALLING_VALUES
    }

    for s_value in STALLING_VALUES:
        row_name = rows[STALLING_TAGS[s_value]][0]

        res = RESULTS[row_name]

        df = object_array_to_df(res, "blackholes_merged")
        mass = df.groupby("time_merged")["mass"].max()

        tk = mass.index
        tv = mass.values

        tt = np.zeros(len(tv))

        lowest = tv[0]

        for i, t_val in enumerate(tv):
            if t_val < lowest:
                tt[i] = lowest
            else:
                tt[i] = t_val
                lowest = t_val

        c = stall_color(s_value, STALLING_VALUES)
        l = ("variable" if s_value == -1 else f"" if s_value == 0 else f"{s_value:g}")

        ax.step(tk / 1000000, tt, lw=1, color=c, label=f"{presc} {l}", ls=ls, where="post")

w = select_with_settings(
    RESULTS,
    stalling_separation=0,
    gas_hardening_prescription="analytical",
    nsc_imf_bh_method="default",
    mu_circ_encounter=0.9,
    mu_ecc_encounter=0.1
)

res = RESULTS[w[0]]

df = object_array_to_df(res, "blackholes_merged")
mass = df.groupby("time_merged")["mass"].max()

tk = mass.index
tv = mass.values

tt = np.zeros(len(tv))

lowest = tv[0]

for i, t_val in enumerate(tv):
    if t_val < lowest:
        tt[i] = lowest
    else:
        tt[i] = t_val
        lowest = t_val

ppl, = ax.step(tk / 1000000, tt, lw=1, color="tab:purple", label=f"_nolegend_", ls=":", where="post")

ax.set_xticks([0, .10, .20, .30, .40, .50, .60, .70])
ax.set_xlabel("time [Myr]", fontsize=7)
ax.set_ylabel("remnant mass [M_sun]", fontsize=7)

second_legend = ax.legend(handles=[ppl], labels=["analytical"], fontsize=4.5, frameon=False, title_fontsize=4.5, bbox_to_anchor=(0.4, 0.7))

ax.legend(fontsize=4.5, frameon=False, title=r"stall [$r_g$]", title_fontsize=4.5, ncol=2)

ax.add_artist(second_legend)

fig.suptitle("Max Remnant Mass vs Time", fontsize=7)